# 연습: 공개 데이터셋(TACO)으로 hw02 파이프라인 미리 돌려 보기

> ⚠️ **제출용이 아닙니다.** 이 노트북은 사진을 찍기 전에 hw02의 코드 흐름(전처리 → 데이터 로더 → 모델 → 학습)을 미리 익히기 위한 연습용입니다.
> 여기서 만든 가중치나 사진은 **제출하지 않고**, 교수님 점수 확인 서버(hw02 8절)에도 **보내지 않습니다.** 그래서 이 노트북에는 hw02의 7절(제출 파일)과 8절(점수 확인)이 없습니다.
> 실제 과제는 내가 찍은 사진으로 `hw02.ipynb`에서 처음부터 다시 학습합니다.

**데이터:** [TACO (Trash Annotations in Context)](http://tacodataset.org) — 실외·실내에서 찍은 쓰레기 사진 1,500장에 물건마다 상자가 그려져 있습니다. ([Hugging Face 사본](https://huggingface.co/datasets/pictograph/taco))
상자 중 아래 클래스만 잘라내서 hw02와 같은 `can / carton / pet` 폴더를 만듭니다.

| hw02 클래스 | TACO 클래스 | 상자 수 |
|---|---|---|
| `can` | Drink can | 229 |
| `carton` | Drink carton | 45 |
| `pet` | Clear plastic bottle, Other plastic bottle | 335 |

종이팩이 적어서 클래스 불균형이 있습니다. 이것도 연습 포인트입니다(클래스별 정확도를 꼭 볼 것).

라이선스: 주석 CC BY 4.0 (Proença & Simões, 2020), 이미지는 개별 라이선스 — **비상업적 교육용으로만** 씁니다.

**Colab 설정:** 런타임 → 런타임 유형 변경 → T4 GPU

In [ ]:
# 준비
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
print(device)

## 0. TACO에서 can / carton / pet 잘라내기 (연습용 추가 코드)

hw02에서는 이 부분이 "드라이브에 내 사진 올리기"입니다. 여기서는 TACO 사진에서 해당 물건 상자를 조금 여유 있게 잘라 `./practice_photos/{can,carton,pet}`에 저장합니다.
원본 사진이 커서(장당 3~4MB) 필요한 약 400장만 내려받는 데 몇 분 걸립니다.

In [ ]:
import os, json, random
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download
from PIL import Image, ImageOps

REPO = 'pictograph/taco'
TACO_TO_HW02 = {'Drink can': 'can', 'Drink carton': 'carton',
                'Clear plastic bottle': 'pet', 'Other plastic bottle': 'pet'}
PHOTO_DIR = './practice_photos'   # hw02의 '/content/drive/MyDrive/hw02' 자리
MIN_BOX = 40          # 너무 작은 상자(짧은 변 40픽셀 미만)는 버린다
PAD = 0.15            # 상자 둘레로 15% 여유를 두고 자른다
MAX_PER_CLASS = 200   # 클래스마다 최대 장수 (불균형을 조금 줄인다)

meta = hf_hub_download(REPO, 'train/metadata.jsonl', repo_type='dataset')
jobs = {}   # 파일 이름 -> [(클래스, 상자), ...]
for line in open(meta):
    r = json.loads(line)
    for box, name in zip(r['objects']['bbox'], r['objects']['category_names']):
        if name in TACO_TO_HW02 and min(box[2], box[3]) >= MIN_BOX:
            jobs.setdefault(r['file_name'], []).append((TACO_TO_HW02[name], box))
print('내려받을 사진', len(jobs), '장')

def crop_one(item):
    file_name, boxes = item
    path = hf_hub_download(REPO, 'train/' + file_name, repo_type='dataset')
    img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    out = []
    for c, (x, y, w, h) in boxes:
        px, py = w * PAD, h * PAD
        crop = img.crop((max(0, x - px), max(0, y - py),
                         min(img.width, x + w + px), min(img.height, y + h + py)))
        s = 384 / min(crop.size)                    # 저장 용량을 줄이려고 짧은 변 384로
        if s < 1:
            crop = crop.resize((round(crop.width * s), round(crop.height * s)))
        out.append((c, crop))
    return file_name, out

for c in set(TACO_TO_HW02.values()):
    os.makedirs(os.path.join(PHOTO_DIR, c), exist_ok=True)
counts = {}
with ThreadPoolExecutor(8) as ex:
    for file_name, crops in ex.map(crop_one, sorted(jobs.items())):
        stem = file_name.replace('/', '_').rsplit('.', 1)[0]
        for k, (c, crop) in enumerate(crops):
            counts[c] = counts.get(c, 0) + 1
            crop.save(os.path.join(PHOTO_DIR, c, f'{stem}_{k}.jpg'), quality=90)

# 클래스마다 최대 MAX_PER_CLASS장만 남긴다
random.seed(0)
for c in counts:
    names = sorted(os.listdir(os.path.join(PHOTO_DIR, c)))
    random.shuffle(names)
    for n in names[MAX_PER_CLASS:]:
        os.remove(os.path.join(PHOTO_DIR, c, n))
print({c: len(os.listdir(os.path.join(PHOTO_DIR, c))) for c in sorted(counts)})

## 2. 사진 줄이고 훈련용·검증용 나누기 — hw02 2절과 같은 코드

hw02의 2절 코드를 거의 그대로 씁니다. 다른 점은 `PHOTO_DIR`가 드라이브 대신 `./practice_photos`이고, 결과 폴더가 `./practice_data`라는 것뿐입니다.

In [ ]:
import os, csv, random, unicodedata
from PIL import Image, ImageOps

CLASSES = ['can', 'carton', 'pet']
DATA = './practice_data'
random.seed(42)
rows = []
for c in CLASSES:
    src = os.path.join(PHOTO_DIR, c)
    assert os.path.isdir(src), f'{src} 폴더가 없습니다.'
    names = sorted(n for n in os.listdir(src) if not n.startswith('.'))
    assert len(names) >= 5, f'{c} 폴더에 사진이 5장 이상 있어야 합니다.'
    random.shuffle(names)
    n_val = len(names) // 5     # 클래스마다 5분의 1은 검증용
    for split in ['train', 'val']:
        os.makedirs(os.path.join(DATA, split, c), exist_ok=True)
    for k, name in enumerate(names):
        img = Image.open(os.path.join(src, name))
        img = ImageOps.exif_transpose(img).convert('RGB')
        w, h = img.size
        scale = 256 / min(w, h)                      # 짧은 변이 256이 되도록
        img = img.resize((round(w * scale), round(h * scale)))
        split = 'val' if k < n_val else 'train'
        file = f'{c}_{k:04d}.jpg'
        img.save(os.path.join(DATA, split, c, file), quality=90)

for split in ['train', 'val']:
    print(split, {c: len(os.listdir(os.path.join(DATA, split, c))) for c in CLASSES})

In [ ]:
# 클래스마다 훈련 사진 여섯 장 보기
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for row, c in zip(axes, CLASSES):
    files = sorted(os.listdir(os.path.join(DATA, 'train', c)))[:6]
    for ax, file in zip(row, files):
        ax.imshow(Image.open(os.path.join(DATA, 'train', c, file)))
        ax.set_title(c)
    for ax in row:
        ax.axis('off')
plt.show()

## 3. 전처리와 데이터 로더

`tf`는 hw02와 똑같습니다(채점기 전처리, 바꾸지 않음).

In [ ]:
from torchvision import transforms

mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

### 3.1 훈련용 전처리 — 빈칸 ① 연습 답안

테스트 사진 조건(멀리서 작게, 위에서, 손에 든 것, 실내·실외 조명, 찌그러짐)에 맞춰 고른 증강입니다.

| 증강 | 왜 |
|---|---|
| `RandomResizedCrop(224, scale=(0.4, 1.0))` | 물건이 크게/작게, 화면 여기저기 나오는 경우. 크기를 224×224로 맞추는 역할도 함 |
| `RandomHorizontalFlip()` | 좌우가 바뀌어도 같은 물건 |
| `RandomRotation(20)` | 비스듬히 들거나 놓인 경우 |
| `ColorJitter(0.3, 0.3, 0.3, 0.05)` | 실내·실외, 그늘·햇빛 등 조명 차이 |
| `ToTensor()`, `Normalize(mean, std)` | 과제 조건: 맨 끝은 이 둘 |

`scale`의 아래쪽 값을 너무 작게(예: 0.08) 하면 물건이 잘려 나간 사진이 많아져 오히려 나빠질 수 있습니다. 바꿔 가며 검증 정확도를 비교해 보세요.

In [ ]:
tf_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.4, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

### 3.2 데이터셋과 로더 — 빈칸 ② 연습 답안

훈련에는 증강(`tf_train`), 검증에는 채점과 같은 `tf`. 훈련 로더만 섞습니다(`shuffle=True`).

In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader

train_ds = datasets.ImageFolder(DATA + '/train', transform=tf_train)
val_ds = datasets.ImageFolder(DATA + '/val', transform=tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
print(train_ds.classes, len(train_ds), len(val_ds))

## 4. 모델 — 빈칸 ③ 연습 답안

- `weights=weights`: ImageNet으로 학습한 가중치를 불러온다
- `requires_grad = False`: 특성 추출부를 모두 동결
- `nn.Linear(512, 3)`: ResNet-18 마지막 특성 512개 → 클래스 3개. 새로 만든 층은 기본이 `requires_grad=True`라 이 층만 학습된다
- 학습할 파라미터 = 512 × 3 + 3 = **1539**

In [ ]:
from torchvision import models
import torch.nn as nn

weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights)
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 3)   # in_features는 512
model = model.to(device)

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(n_train)
assert n_train == 1539

## 5. 학습

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total = 0.0
    n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(yb)
        n += len(yb)
    return total / n

def evaluate(model, loader, criterion):
    model.eval()
    total = 0.0
    correct = 0
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            total += criterion(out, yb).item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            n += len(yb)
    return total / n, correct / n

### 5.2 학습 — 빈칸 ④ 연습 답안

- 옵티마이저에는 학습하는 분류부만: `model.fc.parameters()`
- 에폭 수: 훈련 사진이 약 350장이면 32장씩 11묶음 → 한 에폭에 가중치를 11번 고칩니다. 아래 그래프에서 검증 정확도가 더 오르지 않는 지점을 보고 정합니다.
  내 사진이 이보다 적으면(예: 클래스당 30장 → 훈련 72장 → 3묶음) 에폭을 더 늘려야 합니다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

EPOCHS = 25
history = []
for epoch in range(EPOCHS):
    train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    history.append(val_acc)
    print(epoch, round(val_loss, 4), round(val_acc, 4))

In [ ]:
plt.plot(range(1, len(history) + 1), history, marker='o')
plt.xlabel('epoch')
plt.ylabel('val accuracy')
plt.ylim(0, 1.05)
plt.show()

## 6. 클래스별 정확도와 혼동 행렬 (연습용 추가)

hw02 8절도 클래스별 정확도를 알려 줍니다. 전체 정확도가 높아도 사진이 적은 클래스(여기서는 `carton`)가 낮을 수 있습니다.

In [ ]:
import numpy as np

conf = np.zeros((3, 3), dtype=int)   # conf[실제, 판별]
model.eval()
with torch.no_grad():
    for xb, yb in val_loader:
        pred = model(xb.to(device)).argmax(1).cpu()
        for t, p in zip(yb, pred):
            conf[t, p] += 1
print('행 = 실제, 열 = 판별', CLASSES)
print(conf)
for i, c in enumerate(CLASSES):
    print(f'{c:7s} 정확도 {conf[i, i] / conf[i].sum():.3f}  ({conf[i, i]}/{conf[i].sum()})')

## 7. 틀린 검증 사진 보기 — hw02 6절과 같은 코드

In [ ]:
raw_val = datasets.ImageFolder(DATA + '/val')
model.eval()
wrong = []
for i in range(len(raw_val)):
    img, label = raw_val[i]
    with torch.no_grad():
        p = model(tf(img).unsqueeze(0).to(device)).argmax(1).item()
    if p != label:
        wrong.append((img, label, p))

print('틀린 사진', len(wrong), '/', len(raw_val))
if wrong:
    fig, axes = plt.subplots(1, min(len(wrong), 8), figsize=(16, 2.8), squeeze=False)
    for ax, (img, label, p) in zip(axes[0], wrong):
        ax.imshow(img)
        ax.set_title(f'{CLASSES[label]} → {CLASSES[p]}', color='red')
        ax.axis('off')
    plt.show()

## 8. 튜닝 연습 — 설정을 바꿔 가며 비교하기

같은 데이터로 **한 번에 하나씩만** 바꿔서 검증 정확도를 비교합니다. 내 사진으로 hw02를 할 때도 똑같이 하면 됩니다.

볼 숫자는 두 개입니다.
- **전체 정확도**: 맞힌 장수 / 전체 장수. 사진이 많은 클래스 점수에 끌려갑니다.
- **클래스 평균 정확도**: can·carton·pet 정확도의 평균. 테스트 사진 97장은 세 클래스가 비슷하게 섞여 있을 것이므로 이쪽이 실제 점수에 더 가깝습니다.

주의: 검증 사진이 적어서(특히 종이팩 8장) 1~2장 차이로 숫자가 크게 흔들립니다. 몇 %p 차이는 우연일 수 있습니다.

In [ ]:
# 실험 함수: 설정 하나로 새 모델을 처음부터 학습하고 결과를 기록한다
import numpy as np

results, curves = [], {}

def class_acc(m):
    conf = np.zeros((3, 3), dtype=int)
    m.eval()
    with torch.no_grad():
        for xb, yb in val_loader:
            pred = m(xb.to(device)).argmax(1).cpu()
            for t, p in zip(yb, pred):
                conf[t, p] += 1
    return [conf[i, i] / conf[i].sum() for i in range(3)]

def run(name, train_tf, epochs=30, lr=0.001, class_weight=False):
    torch.manual_seed(42)                       # 모든 실험을 같은 출발점에서
    ds = datasets.ImageFolder(DATA + '/train', transform=train_tf)
    loader = DataLoader(ds, batch_size=32, shuffle=True)
    m = models.resnet18(weights=weights)
    for p in m.parameters():
        p.requires_grad = False
    m.fc = nn.Linear(512, 3)
    m = m.to(device)
    w = None
    if class_weight:                            # 사진이 적은 클래스의 손실을 더 크게
        cnt = torch.bincount(torch.tensor(ds.targets), minlength=3).float()
        w = (cnt.sum() / (3 * cnt)).to(device)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(m.fc.parameters(), lr=lr)
    hist = []
    for e in range(epochs):
        train_one_epoch(m, loader, crit, opt)
        hist.append(evaluate(m, val_loader, nn.CrossEntropyLoss())[1])
    ca = class_acc(m)
    results.append({'설정': name, '전체(마지막)': hist[-1], '전체(최고)': max(hist),
                    '최고 에폭': int(np.argmax(hist)) + 1, 'can': ca[0], 'carton': ca[1],
                    'pet': ca[2], '클래스 평균': float(np.mean(ca))})
    curves[name] = hist
    print(f'{name:22s} 전체 {hist[-1]:.3f} (최고 {max(hist):.3f})  클래스 평균 {np.mean(ca):.3f}  '
          f'can {ca[0]:.2f} carton {ca[1]:.2f} pet {ca[2]:.2f}')

In [ ]:
T = transforms
end = [T.ToTensor(), T.Normalize(mean, std)]

aug_none   = tf                                                       # 증강 없음 (검증과 같은 전처리)
aug_weak   = T.Compose([T.RandomResizedCrop(224, scale=(0.7, 1.0)), T.RandomHorizontalFlip()] + end)
aug_mid    = tf_train                                                 # 3.1에서 고른 증강
aug_strong = T.Compose([T.RandomResizedCrop(224, scale=(0.2, 1.0)), T.RandomHorizontalFlip(),
                        T.RandomRotation(30), T.ColorJitter(0.4, 0.4, 0.4, 0.1),
                        T.RandomGrayscale(0.1)] + end)

run('A 증강 없음',            aug_none)
run('B 약한 증강',            aug_weak)
run('C 중간 증강(3.1)',        aug_mid)
run('D 강한 증강',            aug_strong)
run('E 중간 증강+클래스 가중치', aug_mid, class_weight=True)
run('F 중간 증강, lr 0.003',  aug_mid, lr=0.003)

In [ ]:
# 에폭에 따른 검증 정확도 비교
plt.figure(figsize=(9, 4))
for name, h in curves.items():
    plt.plot(range(1, len(h) + 1), h, label=name)
plt.xlabel('epoch'); plt.ylabel('val accuracy'); plt.ylim(0.3, 1.0); plt.legend(fontsize=8)
plt.show()

for r in sorted(results, key=lambda r: -r['클래스 평균']):
    print(f"{r['설정']:22s} 클래스 평균 {r['클래스 평균']:.3f}  전체 {r['전체(마지막)']:.3f}  최고 {r['전체(최고)']:.3f}@{r['최고 에폭']}")

### 결과 읽는 법과 hw02로 옮기기

- **증강(A~D)**: 가장 좋은 증강을 hw02 빈칸 ①에 그대로 쓴다. 단 TACO와 내 사진은 다르므로, 내 사진으로 A·C·D 정도는 다시 비교해 본다.
- **에폭**: 곡선이 평평해지는 지점이 적당한 `EPOCHS`. 사진 장수가 달라지면 다시 본다.
- **클래스 가중치(E)**: 불균형을 손실로 보정하는 방법. hw02에서는 `criterion` 줄이 제공 코드이므로 바꾸기보다 **사진을 클래스별로 비슷하게 찍는 것**으로 해결하는 게 안전하다.
- **학습률(F)**: 0.001이 제공값. 더 크면 빨리 오르지만 흔들릴 수 있다. hw02에서 `lr=0.001`도 제공 코드이므로 그대로 둔다.

## 실제 과제로 옮길 때

1. 사진을 찍어 드라이브 `내 드라이브/hw02/{can,carton,pet}`에 올린다 (본인 사진만).
2. **`hw02.ipynb`**(교수님 원본)를 Colab에서 열고, 빈칸 ①~④를 이 노트북에서 이해한 내용으로 채운다. 변수 이름은 바꾸지 않는다.
3. `EPOCHS`는 내 사진 장수에 맞춰 다시 정한다(사진이 적을수록 늘린다).
4. 런타임 다시 시작 → 전체 실행 → 8절에서 정확도 확인(하루 5번) → 제출 전 확인 `통과`.

이 노트북에서 만든 가중치·사진은 어디에도 제출하지 않습니다.